# Notebook 04C — Project-Integrated Signal Diagnostics

## Goal

This notebook evaluates the TF-IDF and embedding alpha signals **before** the optimizer.

It uses the existing project modules and paths:

```text
src/config.py
src/data_ingest.py
src/signal_methods.py
src/alpha_model.py
src/signal_diagnostics.py
```

This notebook does **not** replace `run_project.py`.

Instead, it mirrors the early part of `run_project.py`:

```text
load universe
→ add company descriptions
→ compute TF-IDF score
→ compute embedding score
→ build normalized alpha
→ download prices
```

Then it adds the new diagnostic layer:

```text
forward returns
→ monthly IC / rank IC
→ IC summary
→ quantile returns
→ top-minus-bottom spread
→ sector exposure
```

Only after this should you go back to the optimizer and backtest.

## How this relates to `run_project.py`

Your current `run_project.py` already does the full project workflow:

```text
build scores
build alphas
download prices
build optimized weight schedules
run backtests
print performance summaries
```

This notebook stops **before portfolio construction** so you can ask:

```text
Does the alpha signal predict future returns at all?
```

That is different from the optimizer question:

```text
Given alpha and covariance, what weights should we hold?
```

So the order is:

```text
signal diagnostics first
optimizer/backtest second
```

## Cell 1 — Imports

Run this notebook from the repo root in VSCode.

The repo root should contain:

```text
run_project.py
src/
data/
outputs/
```

In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Existing project config.
from src.config import (
    PATH_IN,
    SHEET,
    OUT_DIR,
    THEME_TEXT,
    START_DATE,
    END_DATE,
)

# Existing project data functions.
from src.data_ingest import (
    load_universe,
    add_company_descriptions,
    download_prices,
)

# Existing project signal methods.
from src.signal_methods import (
    add_tfidf_similarity,
    add_embeddings,
    add_embedding_similarity,
)

# Existing project alpha construction.
from src.alpha_model import build_alpha

# New custom baseline signal diagnostics.
from src.signal_diagnostics import (
    build_sector_map_from_universe,
    compute_forward_returns,
    compute_monthly_ic,
    compute_quantile_returns,
    compute_top_bottom_spread,
    compute_sector_exposure,
    summarize_ic,
)

## Cell 2 — Confirm project paths

These paths come from `src/config.py`.

The important ones are:

```text
PATH_IN:
    data/SP500.xlsx

OUT_DIR:
    outputs/

START_DATE, END_DATE:
    Yahoo Finance price window
```

In [ ]:

print("PATH_IN:", PATH_IN)
print("SHEET:", SHEET)
print("OUT_DIR:", OUT_DIR)
print("START_DATE:", START_DATE)
print("END_DATE:", END_DATE)

# Make sure output directory exists.
OUT_DIR.mkdir(exist_ok=True)

## Cell 3 — Output files used by this notebook

The notebook uses cached outputs when available.

This avoids re-running slow Yahoo description calls and OpenAI embedding calls every time.

In [ ]:

SCORED_UNIVERSE_PATH = OUT_DIR / "scored_universe.pkl"
PRICES_PATH = OUT_DIR / "prices.csv"

DIAGNOSTICS_DIR = OUT_DIR / "signal_diagnostics"
DIAGNOSTICS_DIR.mkdir(exist_ok=True)

print("SCORED_UNIVERSE_PATH:", SCORED_UNIVERSE_PATH)
print("PRICES_PATH:", PRICES_PATH)
print("DIAGNOSTICS_DIR:", DIAGNOSTICS_DIR)

## Cell 4 — Runtime switches

Set these depending on what you want to do.

Recommended first run:

```python
REBUILD_SCORES = False
RELOAD_PRICES = False
```

That uses cached files if they exist.

Set `REBUILD_SCORES = True` only if you want to re-run Yahoo descriptions and OpenAI embeddings, which can be slow and may cost money.

In [ ]:

REBUILD_SCORES = False
RELOAD_PRICES = False

print("REBUILD_SCORES:", REBUILD_SCORES)
print("RELOAD_PRICES:", RELOAD_PRICES)

## Cell 5 — Load or build the scored universe

This cell creates the same `df` that `run_project.py` creates before portfolio construction.

Expected columns after this cell:

```text
Ticker
Company Description
tfidf_similarity
alpha_tfidf
embedding_vector
embedding_similarity
alpha_embedding
```

If `outputs/scored_universe.pkl` exists, the notebook loads it.

Otherwise, it builds it from the Excel universe.

In [ ]:

if SCORED_UNIVERSE_PATH.exists() and not REBUILD_SCORES:
    # Fast path:
    # Load prior scored universe created by run_project.py or this notebook.
    df = pd.read_pickle(SCORED_UNIVERSE_PATH)

    print("Loaded cached scored universe:")
    print(SCORED_UNIVERSE_PATH)

else:
    # Slow path:
    # Rebuild the scored universe from the Excel file.
    print("Building scored universe from scratch.")

    # Load universe from the S&P 500 Excel file.
    df = load_universe(PATH_IN, SHEET)

    # Add Yahoo Finance company descriptions.
    df = add_company_descriptions(df)

    # Keep only rows with non-empty descriptions.
    df = df[df["Company Description"].astype(str).str.strip() != ""].copy()

    # Build TF-IDF similarity score.
    df = add_tfidf_similarity(df, THEME_TEXT)

    # Normalize TF-IDF score into alpha.
    df["alpha_tfidf"] = build_alpha(df["tfidf_similarity"])

    # Build OpenAI embeddings for company descriptions.
    df = add_embeddings(df)

    # Build embedding similarity score.
    df = add_embedding_similarity(df, THEME_TEXT)

    # Normalize embedding similarity score into alpha.
    df["alpha_embedding"] = build_alpha(df["embedding_similarity"])

    # Save scored universe so we do not need to rebuild every time.
    df.to_pickle(SCORED_UNIVERSE_PATH)

    print("Saved scored universe:")
    print(SCORED_UNIVERSE_PATH)

print("Rows:", len(df))
print("Columns:", list(df.columns))

display(df.head())

## Cell 6 — Inspect top names by raw similarity score

This is a quick sanity check before diagnostics.

Raw scores are not the same as normalized alpha.

The optimizer and diagnostics should generally use normalized alpha:

```text
alpha_tfidf
alpha_embedding
```

In [ ]:

print("Top 10 names by TF-IDF similarity:")
display(
    df[["Ticker", "tfidf_similarity", "alpha_tfidf"]]
    .sort_values("tfidf_similarity", ascending=False)
    .head(10)
)

print("Top 10 names by embedding similarity:")
display(
    df[["Ticker", "embedding_similarity", "alpha_embedding"]]
    .sort_values("embedding_similarity", ascending=False)
    .head(10)
)

## Cell 7 — Build alpha Series indexed by ticker

This matches the structure already used in `run_project.py`.

Each alpha is:

```text
index = ticker
value = normalized alpha score
```

In [ ]:

alpha_tfidf = df.set_index("Ticker")["alpha_tfidf"].dropna()
alpha_embedding = df.set_index("Ticker")["alpha_embedding"].dropna()

# Normalize ticker labels for safety.
alpha_tfidf.index = alpha_tfidf.index.astype(str).str.strip().str.upper()
alpha_embedding.index = alpha_embedding.index.astype(str).str.strip().str.upper()

print("TF-IDF alpha names:", len(alpha_tfidf))
print("Embedding alpha names:", len(alpha_embedding))

display(alpha_tfidf.head())
display(alpha_embedding.head())

## Cell 8 — Load or download prices using `data_ingest.download_prices`

This is important.

The notebook does **not** rewrite price downloading.

It uses the existing project function:

```python
download_prices(...)
```

Then it saves prices to:

```text
outputs/prices.csv
```

That makes signal diagnostics easier to rerun.

In [ ]:

if PRICES_PATH.exists() and not RELOAD_PRICES:
    # Fast path:
    # Load cached adjusted-close prices.
    prices = pd.read_csv(PRICES_PATH, index_col=0, parse_dates=True)

    print("Loaded cached prices:")
    print(PRICES_PATH)

else:
    # Slow path:
    # Download adjusted-close prices using the existing project function.
    ticker_list = sorted(set(alpha_tfidf.index).union(set(alpha_embedding.index)))

    prices = download_prices(
        ticker_list,
        START_DATE,
        END_DATE,
    )

    # Save prices for repeatable diagnostics.
    prices.to_csv(PRICES_PATH)

    print("Downloaded and saved prices:")
    print(PRICES_PATH)

# Clean basic labels after loading from CSV.
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index()
prices.columns = prices.columns.astype(str).str.strip().str.upper()

print("Price matrix shape:", prices.shape)
display(prices.head())

## Cell 9 — Align alpha and prices

Diagnostics require the same tickers in:

```text
alpha Series
price columns
```

This cell keeps only common names.

In [ ]:

common_tickers = sorted(
    set(prices.columns)
    .intersection(alpha_tfidf.index)
    .intersection(alpha_embedding.index)
)

if len(common_tickers) == 0:
    raise ValueError("No common tickers between prices and alpha signals.")

prices = prices[common_tickers].copy()
alpha_tfidf = alpha_tfidf.loc[common_tickers].copy()
alpha_embedding = alpha_embedding.loc[common_tickers].copy()

print("Common tickers:", len(common_tickers))
print("Prices:", prices.shape)
print("TF-IDF alpha:", alpha_tfidf.shape)
print("Embedding alpha:", alpha_embedding.shape)

## Cell 10 — Compute forward returns

This is the first true signal-diagnostics step.

Forward returns are different from ordinary daily returns.

Daily return:

```text
P_t / P_{t-1} - 1
```

Forward return:

```text
P_{t+h} / P_t - 1
```

Forward returns are what we need to test whether today's alpha predicts future performance.

In [ ]:

forward_returns = compute_forward_returns(
    prices=prices,
    horizons=(21, 63),
)

for horizon, returns in forward_returns.items():
    print(f"{horizon}D forward returns:", returns.shape)

display(forward_returns[21].head())

## Cell 11 — Compute monthly rank IC

The information coefficient is the cross-sectional correlation between:

```text
alpha today
future return
```

We use Spearman by default because it is a rank IC.

This is usually more appropriate for factor research than ordinary Pearson correlation.

In [ ]:

monthly_ic_embedding = compute_monthly_ic(
    alpha=alpha_embedding,
    forward_returns=forward_returns,
    method="spearman",
)

monthly_ic_tfidf = compute_monthly_ic(
    alpha=alpha_tfidf,
    forward_returns=forward_returns,
    method="spearman",
)

print("Embedding monthly IC:")
display(monthly_ic_embedding.head())

print("TF-IDF monthly IC:")
display(monthly_ic_tfidf.head())

monthly_ic_embedding.to_csv(DIAGNOSTICS_DIR / "monthly_ic_embedding.csv")
monthly_ic_tfidf.to_csv(DIAGNOSTICS_DIR / "monthly_ic_tfidf.csv")

## Cell 12 — Summarize IC with optional statsmodels HAC t-stats

`summarize_ic(...)` uses `statsmodels` if available.

The idea is:

```text
IC_t = mean IC + error_t
```

Then use a Newey-West/HAC covariance estimate for the t-stat on mean IC.

If `statsmodels` is not installed, the function falls back to a simpler iid t-stat.

In [ ]:

ic_summary_embedding = summarize_ic(monthly_ic_embedding)
ic_summary_tfidf = summarize_ic(monthly_ic_tfidf)

print("Embedding IC summary:")
display(ic_summary_embedding)

print("TF-IDF IC summary:")
display(ic_summary_tfidf)

ic_summary_embedding.to_csv(
    DIAGNOSTICS_DIR / "ic_summary_embedding.csv",
    index=False,
)

ic_summary_tfidf.to_csv(
    DIAGNOSTICS_DIR / "ic_summary_tfidf.csv",
    index=False,
)

## Cell 13 — Plot monthly IC

This plot is a quick visual check.

A useful signal should not necessarily have positive IC every month, but you want to know whether the average is meaningfully positive and whether it is stable or noisy.

In [ ]:

ax = monthly_ic_embedding.plot(
    figsize=(10, 5),
    title="Monthly Rank IC — Embedding Alpha",
)

ax.axhline(0.0, linestyle="--", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Rank IC")
plt.tight_layout()
plt.show()

ax = monthly_ic_tfidf.plot(
    figsize=(10, 5),
    title="Monthly Rank IC — TF-IDF Alpha",
)

ax.axhline(0.0, linestyle="--", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Rank IC")
plt.tight_layout()
plt.show()

## Cell 14 — Compute quantile returns

This groups stocks by alpha score.

For five quantiles:

```text
quantile 1 = lowest alpha names
quantile 5 = highest alpha names
```

A useful long signal should generally show higher future returns in higher alpha quantiles.

In [ ]:

quantile_returns_embedding = compute_quantile_returns(
    alpha=alpha_embedding,
    forward_returns=forward_returns,
    n_quantiles=5,
)

quantile_returns_tfidf = compute_quantile_returns(
    alpha=alpha_tfidf,
    forward_returns=forward_returns,
    n_quantiles=5,
)

print("Embedding quantile returns:")
display(quantile_returns_embedding.head(10))

print("TF-IDF quantile returns:")
display(quantile_returns_tfidf.head(10))

quantile_returns_embedding.to_csv(
    DIAGNOSTICS_DIR / "quantile_returns_embedding.csv",
    index=False,
)

quantile_returns_tfidf.to_csv(
    DIAGNOSTICS_DIR / "quantile_returns_tfidf.csv",
    index=False,
)

## Cell 15 — Average quantile returns

This is the basic monotonicity check.

If the signal works, the average return by quantile should often slope upward from low alpha to high alpha.

In [ ]:

avg_quantile_returns_embedding = (
    quantile_returns_embedding
    .groupby(["horizon", "quantile"])["mean_forward_return"]
    .mean()
    .reset_index()
)

avg_quantile_returns_tfidf = (
    quantile_returns_tfidf
    .groupby(["horizon", "quantile"])["mean_forward_return"]
    .mean()
    .reset_index()
)

print("Embedding average quantile returns:")
display(avg_quantile_returns_embedding)

print("TF-IDF average quantile returns:")
display(avg_quantile_returns_tfidf)

avg_quantile_returns_embedding.to_csv(
    DIAGNOSTICS_DIR / "avg_quantile_returns_embedding.csv",
    index=False,
)

avg_quantile_returns_tfidf.to_csv(
    DIAGNOSTICS_DIR / "avg_quantile_returns_tfidf.csv",
    index=False,
)

## Cell 16 — Plot average quantile returns

This compares low-alpha and high-alpha buckets.

In [ ]:

for horizon in sorted(avg_quantile_returns_embedding["horizon"].unique()):
    data = avg_quantile_returns_embedding.loc[
        avg_quantile_returns_embedding["horizon"] == horizon
    ]

    ax = data.plot(
        x="quantile",
        y="mean_forward_return",
        kind="bar",
        legend=False,
        figsize=(7, 4),
        title=f"Embedding Alpha: Average Forward Return by Quantile ({horizon}D)",
    )

    ax.set_xlabel("Alpha quantile")
    ax.set_ylabel("Mean forward return")
    plt.tight_layout()
    plt.show()

for horizon in sorted(avg_quantile_returns_tfidf["horizon"].unique()):
    data = avg_quantile_returns_tfidf.loc[
        avg_quantile_returns_tfidf["horizon"] == horizon
    ]

    ax = data.plot(
        x="quantile",
        y="mean_forward_return",
        kind="bar",
        legend=False,
        figsize=(7, 4),
        title=f"TF-IDF Alpha: Average Forward Return by Quantile ({horizon}D)",
    )

    ax.set_xlabel("Alpha quantile")
    ax.set_ylabel("Mean forward return")
    plt.tight_layout()
    plt.show()

## Cell 17 — Compute top-minus-bottom spread

This asks:

```text
Did the highest-alpha quantile outperform the lowest-alpha quantile?
```

The spread is:

```text
quantile 5 return - quantile 1 return
```

In [ ]:

spread_embedding = compute_top_bottom_spread(
    quantile_returns_embedding
)

spread_tfidf = compute_top_bottom_spread(
    quantile_returns_tfidf
)

print("Embedding top-minus-bottom spread:")
display(spread_embedding.head())

print("TF-IDF top-minus-bottom spread:")
display(spread_tfidf.head())

spread_embedding.to_csv(DIAGNOSTICS_DIR / "top_bottom_spread_embedding.csv")
spread_tfidf.to_csv(DIAGNOSTICS_DIR / "top_bottom_spread_tfidf.csv")

## Cell 18 — Plot top-minus-bottom spread

Positive values mean top alpha names beat bottom alpha names over the forward return horizon.

In [ ]:

ax = spread_embedding.plot(
    figsize=(10, 5),
    title="Top-minus-Bottom Spread — Embedding Alpha",
)

ax.axhline(0.0, linestyle="--", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Forward return spread")
plt.tight_layout()
plt.show()

ax = spread_tfidf.plot(
    figsize=(10, 5),
    title="Top-minus-Bottom Spread — TF-IDF Alpha",
)

ax.axhline(0.0, linestyle="--", linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Forward return spread")
plt.tight_layout()
plt.show()

## Cell 19 — Build sector map from the same universe DataFrame

This uses the sector information from the S&P 500 Excel file, if available.

The helper searches common column names:

```text
Sector
GICS Sector
GICS_Sector
sector
gics_sector
```

If your Excel file uses a different column name, pass it explicitly.

In [ ]:

try:
    sector_map = build_sector_map_from_universe(df)

    print("Sector map built.")
    print("Sectors:", sector_map.nunique())

    display(sector_map.head())

except Exception as e:
    sector_map = None

    print("Could not build sector map automatically.")
    print("Error:", repr(e))
    print("If the Excel file has a sector column, pass sector_col explicitly.")

## Cell 20 — Compute sector exposure

This checks whether the alpha signal is mostly a sector bet.

For example, an AI infrastructure signal may load heavily on Information Technology.

In [ ]:

if sector_map is not None:
    sector_exposure_embedding = compute_sector_exposure(
        alpha_embedding,
        sector_map,
    )

    sector_exposure_tfidf = compute_sector_exposure(
        alpha_tfidf,
        sector_map,
    )

    print("Embedding sector exposure:")
    display(sector_exposure_embedding)

    print("TF-IDF sector exposure:")
    display(sector_exposure_tfidf)

    sector_exposure_embedding.to_csv(
        DIAGNOSTICS_DIR / "sector_exposure_embedding.csv"
    )

    sector_exposure_tfidf.to_csv(
        DIAGNOSTICS_DIR / "sector_exposure_tfidf.csv"
    )

else:
    print("No sector exposure computed because sector_map is unavailable.")

## Cell 21 — Plot sector exposure

The `abs_alpha_share` column shows where the magnitude of the signal is concentrated.

In [ ]:

if sector_map is not None:
    plot_data = sector_exposure_embedding.sort_values(
        "abs_alpha_share",
        ascending=False,
    )

    ax = plot_data["abs_alpha_share"].plot(
        kind="bar",
        figsize=(9, 4),
        title="Embedding Alpha: Share of Absolute Alpha by Sector",
    )

    ax.set_xlabel("Sector")
    ax.set_ylabel("Absolute alpha share")
    plt.tight_layout()
    plt.show()

    plot_data = sector_exposure_tfidf.sort_values(
        "abs_alpha_share",
        ascending=False,
    )

    ax = plot_data["abs_alpha_share"].plot(
        kind="bar",
        figsize=(9, 4),
        title="TF-IDF Alpha: Share of Absolute Alpha by Sector",
    )

    ax.set_xlabel("Sector")
    ax.set_ylabel("Absolute alpha share")
    plt.tight_layout()
    plt.show()

else:
    print("No sector exposure plot because sector_map is unavailable.")

## Cell 22 — Compact diagnostics summary

This gives a quick comparison between embedding alpha and TF-IDF alpha.

In [ ]:

diagnostic_summary = pd.DataFrame(
    {
        "embedding_mean_ic": monthly_ic_embedding.mean(),
        "tfidf_mean_ic": monthly_ic_tfidf.mean(),
        "embedding_mean_spread": spread_embedding.mean(),
        "tfidf_mean_spread": spread_tfidf.mean(),
    }
)

display(diagnostic_summary)

diagnostic_summary.to_csv(
    DIAGNOSTICS_DIR / "diagnostic_summary.csv"
)

## Cell 23 — What to do after this notebook

If the diagnostics look weak:

```text
do not trust optimizer results as evidence of alpha quality
```

If the diagnostics look promising:

```text
continue to optimizer/backtest
```

The next project step is your existing portfolio code:

```text
build_all_weight_schedules(...)
run_all_backtests(...)
compute_perf_stats(...)
```

That step answers a different question:

```text
Given an alpha and a risk model, what portfolio performance results?
```

## Cell 24 — Optional: code block to add to `run_project.py`

Once this notebook works, copy the diagnostics block into `run_project.py`
right after:

```python
alpha_tfidf = df.set_index("Ticker")["alpha_tfidf"].dropna()
alpha_embedding = df.set_index("Ticker")["alpha_embedding"].dropna()
```

and before:

```python
schedules = build_all_weight_schedules(...)
```

In [ ]:

# Optional run_project.py insertion:
#
# from src.signal_diagnostics import (
#     build_sector_map_from_universe,
#     compute_forward_returns,
#     compute_monthly_ic,
#     compute_quantile_returns,
#     compute_top_bottom_spread,
#     compute_sector_exposure,
#     summarize_ic,
# )
#
# diagnostics_dir = OUT_DIR / "signal_diagnostics"
# diagnostics_dir.mkdir(exist_ok=True)
#
# prices.to_csv(OUT_DIR / "prices.csv")
#
# forward_returns = compute_forward_returns(prices, horizons=(21, 63))
#
# monthly_ic_embedding = compute_monthly_ic(alpha_embedding, forward_returns)
# monthly_ic_tfidf = compute_monthly_ic(alpha_tfidf, forward_returns)
#
# monthly_ic_embedding.to_csv(diagnostics_dir / "monthly_ic_embedding.csv")
# monthly_ic_tfidf.to_csv(diagnostics_dir / "monthly_ic_tfidf.csv")
#
# summarize_ic(monthly_ic_embedding).to_csv(
#     diagnostics_dir / "ic_summary_embedding.csv",
#     index=False,
# )
#
# summarize_ic(monthly_ic_tfidf).to_csv(
#     diagnostics_dir / "ic_summary_tfidf.csv",
#     index=False,
# )
#
# qret_embedding = compute_quantile_returns(alpha_embedding, forward_returns)
# qret_tfidf = compute_quantile_returns(alpha_tfidf, forward_returns)
#
# qret_embedding.to_csv(
#     diagnostics_dir / "quantile_returns_embedding.csv",
#     index=False,
# )
#
# qret_tfidf.to_csv(
#     diagnostics_dir / "quantile_returns_tfidf.csv",
#     index=False,
# )
#
# compute_top_bottom_spread(qret_embedding).to_csv(
#     diagnostics_dir / "top_bottom_spread_embedding.csv",
# )
#
# compute_top_bottom_spread(qret_tfidf).to_csv(
#     diagnostics_dir / "top_bottom_spread_tfidf.csv",
# )
#
# try:
#     sector_map = build_sector_map_from_universe(df)
#
#     compute_sector_exposure(alpha_embedding, sector_map).to_csv(
#         diagnostics_dir / "sector_exposure_embedding.csv",
#     )
#
#     compute_sector_exposure(alpha_tfidf, sector_map).to_csv(
#         diagnostics_dir / "sector_exposure_tfidf.csv",
#     )
#
# except Exception as e:
#     print("Skipped sector diagnostics:", repr(e))